# Clase 25: Ajuste de Hiperparámetros y Selección de Modelos

**Idea central:** El ajuste de hiperparámetros es como calibrar un instrumento — pequeños cambios en la configuración pueden mejorar significativamente los resultados.

En esta clase aprenderemos a:
- Distinguir parámetros de hiperparámetros
- Usar GridSearchCV y RandomizedSearchCV
- Interpretar curvas de aprendizaje y validación

In [ ]:
# Celda 1: Imports y carga de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Cargar dataset de estudiantes
df = pd.read_csv('../../datasets/estudiantes.csv')
print('Forma del dataset:', df.shape)
print('\nColumnas:', df.columns.tolist())
print('\nPrimeras filas:')
df.head()

In [ ]:
# Celda 2: Preparar datos y modelo base (sin optimizar)
# Separamos features y target
X = df.drop(columns=['aprobado'])
y = df['aprobado']

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')

# Modelo base con parámetros por defecto
modelo_base = RandomForestClassifier(random_state=42)
modelo_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, modelo_base.predict(X_test))
print(f'\nAccuracy base: {acc_base:.4f}')
print(f'n_estimators por defecto: {modelo_base.n_estimators}')
print(f'max_depth por defecto: {modelo_base.max_depth}  (None = sin límite)')

In [ ]:
# Celda 3: GridSearchCV — buscar la mejor combinación de hiperparámetros
from sklearn.model_selection import GridSearchCV

# Definir el grid (todas las combinaciones a probar)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None]
}

# Total de combinaciones: 3 x 4 = 12
# Con cv=5: 12 x 5 = 60 modelos entrenados en total
print(f'Combinaciones a probar: {3 * 4}')
print(f'Modelos entrenados en total (con cv=5): {3 * 4 * 5}')

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('\nMejores parámetros:', grid_search.best_params_)
print(f'Mejor score CV: {grid_search.best_score_:.4f}')
print(f'Score en test: {grid_search.score(X_test, y_test):.4f}')

In [ ]:
# Celda 4: Explorar todos los resultados del GridSearch
resultados_grid = pd.DataFrame(grid_search.cv_results_)

# Las 5 mejores combinaciones
top5 = resultados_grid.sort_values('rank_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].head(5)

print('Las 5 mejores combinaciones:')
print(top5.to_string())

# Heatmap de resultados
scores_matrix = resultados_grid.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

import seaborn as sns
plt.figure(figsize=(8, 5))
sns.heatmap(scores_matrix, annot=True, fmt='.4f', cmap='YlOrRd')
plt.title('Accuracy por combinación de hiperparámetros (GridSearchCV)')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.tight_layout()
plt.show()

In [ ]:
# Celda 5: RandomizedSearchCV — más rápido para espacios grandes
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Distribuciones de hiperparámetros (mucho más amplio que el grid)
param_dist = {
    'n_estimators': randint(50, 400),
    'max_depth': randint(2, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,  # probar solo 30 combinaciones aleatorias
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print('Mejores parámetros (RandomizedSearch):', random_search.best_params_)
print(f'Mejor score CV: {random_search.best_score_:.4f}')
print(f'Score en test: {random_search.score(X_test, y_test):.4f}')

# Comparación
print('\n--- Comparación ---')
print(f'Modelo base:           {acc_base:.4f}')
print(f'GridSearchCV:          {grid_search.score(X_test, y_test):.4f}')
print(f'RandomizedSearchCV:    {random_search.score(X_test, y_test):.4f}')

In [ ]:
# Celda 6: Curva de validación — efecto de max_depth
from sklearn.model_selection import validation_curve

param_range = [2, 5, 10, 15, 20, 30, 50]

train_scores_vc, val_scores_vc = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    param_name='max_depth',
    param_range=param_range,
    cv=5,
    scoring='accuracy'
)

plt.figure(figsize=(10, 5))
plt.plot(param_range, train_scores_vc.mean(axis=1), 'o-', color='steelblue', label='Entrenamiento')
plt.fill_between(param_range,
                 train_scores_vc.mean(axis=1) - train_scores_vc.std(axis=1),
                 train_scores_vc.mean(axis=1) + train_scores_vc.std(axis=1),
                 alpha=0.15, color='steelblue')
plt.plot(param_range, val_scores_vc.mean(axis=1), 'o-', color='darkorange', label='Validación')
plt.fill_between(param_range,
                 val_scores_vc.mean(axis=1) - val_scores_vc.std(axis=1),
                 val_scores_vc.mean(axis=1) + val_scores_vc.std(axis=1),
                 alpha=0.15, color='darkorange')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Curva de validación: efecto de max_depth en RandomForest')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('Observa: cuando max_depth es muy alto, el entrenamiento sube pero la validación baja = overfitting')

In [ ]:
# Celda 7: Curva de aprendizaje — ¿más datos ayudarían?
from sklearn.model_selection import learning_curve

best_params = grid_search.best_params_

train_sizes, train_scores_lc, val_scores_lc = learning_curve(
    RandomForestClassifier(**best_params, random_state=42),
    X, y,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores_lc.mean(axis=1), 'o-', color='steelblue', label='Entrenamiento')
plt.fill_between(train_sizes,
                 train_scores_lc.mean(axis=1) - train_scores_lc.std(axis=1),
                 train_scores_lc.mean(axis=1) + train_scores_lc.std(axis=1),
                 alpha=0.15, color='steelblue')
plt.plot(train_sizes, val_scores_lc.mean(axis=1), 'o-', color='darkorange', label='Validación')
plt.fill_between(train_sizes,
                 val_scores_lc.mean(axis=1) - val_scores_lc.std(axis=1),
                 val_scores_lc.mean(axis=1) + val_scores_lc.std(axis=1),
                 alpha=0.15, color='darkorange')
plt.xlabel('Número de muestras de entrenamiento')
plt.ylabel('Accuracy')
plt.title('Curva de aprendizaje — ¿se beneficia el modelo de más datos?')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Celda 8: Modelo final con los mejores parámetros
mejor_modelo = RandomForestClassifier(**grid_search.best_params_, random_state=42)
mejor_modelo.fit(X_train, y_train)

y_pred = mejor_modelo.predict(X_test)

print('=== Evaluación del modelo optimizado ===')
print(classification_report(y_test, y_pred))

acc_optimizado = accuracy_score(y_test, y_pred)
mejora = (acc_optimizado - acc_base) * 100
print(f'Accuracy base:       {acc_base:.4f}')
print(f'Accuracy optimizado: {acc_optimizado:.4f}')
print(f'Mejora:              +{mejora:.2f}%')

In [ ]:
# Celda 9: Importancia de features del modelo optimizado
importancias = pd.Series(
    mejor_modelo.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importancias.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title(f'Importancia de features — RandomForest optimizado\nMejores params: {grid_search.best_params_}')
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

## Resumen de la clase

| Concepto | Descripción |
|---|---|
| **Hiperparámetro** | Configuración que definimos antes de entrenar |
| **Parámetro** | Valor que el modelo aprende durante el entrenamiento |
| **GridSearchCV** | Prueba todas las combinaciones del grid especificado |
| **RandomizedSearchCV** | Prueba combinaciones aleatorias — más rápido para espacios grandes |
| **cv=5** | Cada combinación se evalúa con 5-fold cross-validation |
| **best_params_** | La mejor combinación encontrada |
| **Curva de validación** | Muestra el efecto de un hiperparámetro en el rendimiento |
| **Curva de aprendizaje** | Muestra si más datos mejorarían el modelo |

**Regla de oro:** El test set solo se usa UNA VEZ, al final, para la evaluación definitiva.